# SOTA Baseline: ThermalFusion-Net (2023) on FLAME Dataset

Purpose: Train ThermalFusion-Net as a DICTA 2026 baseline using FLAME segmentation data.

Key properties:
- Shared FLAMEDataset class
- Thermal-prioritized residual fusion at 1/4, 1/8, 1/16
- AdamW + OneCycleLR
- Benchmarked outputs: Params (M), FLOPs (G), P95 Latency (ms)

## 1. Setup Imports and Paths

In [1]:
import json
import time
import warnings
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent.parent
import sys
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'scripts' / 'data'))
sys.path.insert(0, str(PROJECT_ROOT / 'models' / 'sota_baselines'))

from flame_dataset import FLAMEDataset
from thermalfusion_net_2023 import create_thermalfusion_net_2023

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device: {device}')
print(f'Project root: {PROJECT_ROOT}')

Device: cpu
Project root: c:\SPJAIN\BushFire-Detection


## 2. Configuration

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'Output' / 'Segmentation_Augmented'
CKPT_DIR = PROJECT_ROOT / 'models' / 'trained' / 'sota_baselines' / 'thermalfusion_net_2023'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT_PATH = CKPT_DIR / 'best_model.pth'
METRICS_OUT = CKPT_DIR / 'sota_thermalfusion_net_metrics.json'

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 35
NUM_WORKERS = 0
MAX_LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 10

print('ThermalFusion-Net Training Configuration')
print(f'DATA_DIR exists: {DATA_DIR.exists()}')
print(f'Checkpoint path: {BEST_CKPT_PATH}')
print(f'Metrics JSON path: {METRICS_OUT}')
print(f'Target mIoU: ~0.81 | Target model size: <8MB')

ThermalFusion-Net Training Configuration
DATA_DIR exists: True
Checkpoint path: c:\SPJAIN\BushFire-Detection\models\trained\sota_baselines\thermalfusion_net_2023\best_model.pth
Target mIoU: ~0.81 | Target model size: <8MB


## 3. Dataset and DataLoaders (Shared FLAMEDataset)

In [3]:
train_dataset = FLAMEDataset(
    root_dir=DATA_DIR,
    split='train',
    img_size=IMG_SIZE,
    train_ratio=0.8,
    augment=True,
)

val_dataset = FLAMEDataset(
    root_dir=DATA_DIR,
    split='val',
    img_size=IMG_SIZE,
    train_ratio=0.8,
    augment=False,
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'), drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda')
)

sample = next(iter(train_loader))
print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')
print(f'Image tensor shape: {sample["image"].shape}')
print('Note: FLAMEDataset applies ImageNet normalization to RGB. Thermal cue is derived inside the model from de-normalized RGB to preserve fire-intensity cues.')

FLAME Dataset (train): 646 samples, size=256x256, augment=True
FLAME Dataset (val): 162 samples, size=256x256, augment=False
Train samples: 646 | Val samples: 162
Image tensor shape: torch.Size([16, 3, 256, 256])
Note: FLAMEDataset applies ImageNet normalization to RGB. Thermal cue is derived inside the model from de-normalized RGB to preserve fire-intensity cues.


## 4. Build ThermalFusion-Net (2023)

In [4]:
model = create_thermalfusion_net_2023(in_channels=3, out_channels=1, base_channels=12).to(device)
stats = model.get_parameter_count()
print(f'Total params: {stats["total_parameters"]:,}')
print(f'Model size: {stats["model_size_mb"]:.2f} MB')

x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    y = model(x)
print(f'Forward output shape: {y.shape}')

Total params: 1,647,901
Model size: 6.29 MB
Forward output shape: torch.Size([2, 1, 256, 256])


## 5. Loss, Metrics, Optimizer, OneCycleLR

In [5]:
bce_loss = nn.BCELoss()

def dice_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    pred = pred.view(-1)
    target = target.view(-1)
    inter = (pred * target).sum()
    return 1.0 - (2.0 * inter + eps) / (pred.sum() + target.sum() + eps)

def combined_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return 0.5 * bce_loss(pred, target) + 0.5 * dice_loss(pred, target)

def compute_metrics(pred: torch.Tensor, target: torch.Tensor, thr: float = 0.5) -> Dict[str, float]:
    pb = (pred > thr).float()
    tb = target.float()
    tp = (pb * tb).sum(dim=[2, 3])
    fp = (pb * (1 - tb)).sum(dim=[2, 3])
    fn = ((1 - pb) * tb).sum(dim=[2, 3])
    iou = tp / (tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)
    return {'mIoU': float(iou.mean().item()), 'F1': float(f1.mean().item())}

optimizer = optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
scheduler = OneCycleLR(
    optimizer=optimizer,
    max_lr=MAX_LR,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.2,
    div_factor=10.0,
    final_div_factor=100.0,
)

print('AdamW + OneCycleLR initialized')

AdamW + OneCycleLR initialized


## 6. Train and Save Best Checkpoint

In [6]:
best_miou = 0.0
best_epoch = 0
history = {'train_loss': [], 'val_miou': [], 'val_f1': []}
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0

    for batch in train_loader:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)

        optimizer.zero_grad()
        pred = model(images)
        loss = combined_loss(pred, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        train_loss_sum += float(loss.item())

    avg_train_loss = train_loss_sum / max(len(train_loader), 1)

    model.eval()
    val_ious = []
    val_f1s = []
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            pred = model(images)
            m = compute_metrics(pred, masks)
            val_ious.append(m['mIoU'])
            val_f1s.append(m['F1'])

    avg_miou = float(np.mean(val_ious)) if val_ious else 0.0
    avg_f1 = float(np.mean(val_f1s)) if val_f1s else 0.0

    history['train_loss'].append(avg_train_loss)
    history['val_miou'].append(avg_miou)
    history['val_f1'].append(avg_f1)

    if avg_miou > best_miou:
        best_miou = avg_miou
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_miou': best_miou,
            'best_f1': avg_f1,
            'config': {
                'img_size': IMG_SIZE,
                'batch_size': BATCH_SIZE,
                'max_lr': MAX_LR,
                'weight_decay': WEIGHT_DECAY,
            }
        }, BEST_CKPT_PATH)
    else:
        patience_counter += 1

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch + 1:02d}/{EPOCHS} | train_loss={avg_train_loss:.4f} | val_mIoU={avg_miou:.4f} | val_F1={avg_f1:.4f}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch + 1}')
        break

print(f'Best checkpoint saved: {BEST_CKPT_PATH}')
print(f'Best epoch: {best_epoch + 1} | Best mIoU: {best_miou:.4f}')

Epoch 01/35 | train_loss=0.7921 | val_mIoU=0.0354 | val_F1=0.0653
Epoch 05/35 | train_loss=0.7227 | val_mIoU=0.1955 | val_F1=0.3175
Epoch 10/35 | train_loss=0.5813 | val_mIoU=0.3554 | val_F1=0.5024
Epoch 15/35 | train_loss=0.5206 | val_mIoU=0.3228 | val_F1=0.4806
Early stopping at epoch 19
Best checkpoint saved: c:\SPJAIN\BushFire-Detection\models\trained\sota_baselines\thermalfusion_net_2023\best_model.pth
Best epoch: 9 | Best mIoU: 0.4153


## 7. Benchmark: Params (M), FLOPs (G), P95 Latency (ms)

In [6]:
checkpoint = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state'])
model.eval()

def estimate_flops_g(model: nn.Module, input_shape: Tuple[int, int, int, int]) -> float:
    hooks = []
    total_ops = {'v': 0}

    def conv_hook(m, inp, out):
        if not isinstance(m, nn.Conv2d):
            return
        out_h, out_w = out.shape[-2], out.shape[-1]
        kernel_ops = m.kernel_size[0] * m.kernel_size[1] * (m.in_channels / m.groups)
        ops = out_h * out_w * m.out_channels * kernel_ops
        total_ops['v'] += ops

    def linear_hook(m, inp, out):
        if not isinstance(m, nn.Linear):
            return
        total_ops['v'] += m.in_features * m.out_features

    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            hooks.append(module.register_forward_hook(conv_hook))
        elif isinstance(module, nn.Linear):
            hooks.append(module.register_forward_hook(linear_hook))

    x = torch.randn(*input_shape).to(device)
    with torch.no_grad():
        _ = model(x)

    for h in hooks:
        h.remove()

    return float(total_ops['v'] / 1e9)

def benchmark_latency_p95(model: nn.Module, input_shape=(1, 3, 256, 256), warmup=20, runs=100) -> Dict[str, float]:
    x = torch.randn(*input_shape).to(device)
    times = []

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(x)

        for _ in range(runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(x)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append((t1 - t0) * 1000.0)

    arr = np.array(times)
    return {
        'mean_ms': float(arr.mean()),
        'p95_ms': float(np.percentile(arr, 95)),
        'p99_ms': float(np.percentile(arr, 99)),
    }

params_m = stats['total_parameters'] / 1e6
flops_g = estimate_flops_g(model, (1, 3, IMG_SIZE, IMG_SIZE))
lat = benchmark_latency_p95(model, input_shape=(1, 3, IMG_SIZE, IMG_SIZE), warmup=20, runs=100)

print(f'Params (M): {params_m:.3f}')
print(f'FLOPs (G): {flops_g:.3f}')
print(f'Latency P95 (ms): {lat["p95_ms"]:.2f}')

Params (M): 1.648
FLOPs (G): 2.344
Latency P95 (ms): 52.31


## 8. Final Validation and DICTA-Style Export

In [8]:
model.eval()
vals_miou = []
vals_f1 = []

with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)
        pred = model(images)
        m = compute_metrics(pred, masks)
        vals_miou.append(m['mIoU'])
        vals_f1.append(m['F1'])

final_miou = float(np.mean(vals_miou)) if vals_miou else 0.0
final_f1 = float(np.mean(vals_f1)) if vals_f1 else 0.0

results = {
    'model_name': 'ThermalFusion-Net (2023)',
    'dataset': 'FLAME Augmented',
    'checkpoint_path': str(BEST_CKPT_PATH),
    'targets': {
        'miou_target': 0.81,
        'model_size_mb_max': 8.0
    },
    'metrics': {
        'miou': final_miou,
        'f1': final_f1,
        'params_m': float(params_m),
        'flops_g': float(flops_g),
        'latency_p95_ms': float(lat['p95_ms']),
        'latency_p99_ms': float(lat['p99_ms']),
        'latency_mean_ms': float(lat['mean_ms'])
    },
    'training': {
        'optimizer': 'AdamW',
        'weight_decay': WEIGHT_DECAY,
        'scheduler': 'OneCycleLR',
        'epochs_planned': EPOCHS,
        'epochs_run': len(history['train_loss'])
    },
    'dataset_split': {
        'train_samples': len(train_dataset),
        'val_samples': len(val_dataset),
        'total_samples': len(train_dataset) + len(val_dataset)
    }
}

METRICS_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(METRICS_OUT, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print('ThermalFusion-Net results saved')
print(f'Metrics JSON: {METRICS_OUT}')
print(json.dumps(results, indent=2))

ThermalFusion-Net results saved
Metrics JSON: c:\SPJAIN\BushFire-Detection\data\processed\Output\sota_thermalfusion_net_metrics.json
{
  "model_name": "ThermalFusion-Net (2023)",
  "dataset": "FLAME Augmented",
  "checkpoint_path": "c:\\SPJAIN\\BushFire-Detection\\models\\trained\\sota_baselines\\thermalfusion_net_2023\\best_model.pth",
  "targets": {
    "miou_target": 0.81,
    "model_size_mb_max": 8.0
  },
  "metrics": {
    "miou": 0.4152782749045979,
    "f1": 0.5644999254833568,
    "params_m": 1.647901,
    "flops_g": 2.34356736,
    "latency_p95_ms": 74.67532997688977,
    "latency_p99_ms": 116.14376402256626,
    "latency_mean_ms": 70.3677829989465
  },
  "training": {
    "optimizer": "AdamW",
    "weight_decay": 0.0001,
    "scheduler": "OneCycleLR",
    "epochs_planned": 35,
    "epochs_run": 19
  },
  "dataset_split": {
    "train_samples": 646,
    "val_samples": 162,
    "total_samples": 808
  }
}


In [9]:
import pandas as pd

row = {
    'Model': 'ThermalFusion-Net (2023)',
    'Type': 'SOTA Baseline',
    'Dataset': 'FLAME',
    'Params (M)': f'{params_m:.3f}',
    'FLOPs (G)': f'{flops_g:.3f}',
    'Size (MB)': f'{stats["model_size_mb"]:.2f}',
    'mIoU': f'{final_miou:.4f}',
    'F1': f'{final_f1:.4f}',
    'P95 Latency (ms)': f'{lat["p95_ms"]:.2f}',
}

df = pd.DataFrame([row])
print('=' * 110)
print('DICTA BENCHMARK TABLE ROW')
print('=' * 110)
print(df.to_string(index=False))

DICTA BENCHMARK TABLE ROW
                   Model          Type Dataset Params (M) FLOPs (G) Size (MB)   mIoU     F1 P95 Latency (ms)
ThermalFusion-Net (2023) SOTA Baseline   FLAME      1.648     2.344      6.29 0.4153 0.5645            74.68


## 9. Strict Locked-Split Test Evaluation (Machine-Agnostic)

Single-reference evaluation for cross-machine comparability.

This section:
- Loads train/val/test only from the locked repository split JSON.
- Creates the locked split once only if missing, then reuses it.
- Uses deterministic settings and repository-relative paths.
- Runs test-only evaluation on the locked test split.
- Reports full-test aggregated metrics (not batch-averaged).

In [3]:
import hashlib
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

# Deterministic and reproducible execution
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


def resolve_project_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'README.md').exists() and (p / 'data').exists() and (p / 'models').exists():
            return p
    raise FileNotFoundError('Could not resolve PROJECT_ROOT from current notebook/workspace path.')


PROJECT_ROOT = resolve_project_root(Path.cwd())
DATASET_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'Output' / 'Segmentation_Augmented'
STRICT_SPLIT_FILE = PROJECT_ROOT / 'data' / 'processed' / 'Output' / 'DICTA' / 'flame_strict_split_seed42.json'
STRICT_CKPT_PATH = PROJECT_ROOT / 'models' / 'trained' / 'sota_baselines' / 'thermalfusion_net_2023' / 'best_model.pth'
STRICT_METRICS_OUT = STRICT_CKPT_PATH.parent / 'sota_thermalfusion_net_metrics.json'

IMAGE_SIZE = 256
BATCH_SIZE_STRICT = 16
NUM_WORKERS = 0
NORMALIZATION_MEAN = [0.485, 0.456, 0.406]
NORMALIZATION_STD = [0.229, 0.224, 0.225]

# If this same cell is copied into shared-backbone notebook, use DICTA shared-backbone checkpoint.
if Path.cwd().name == '02_training' and 'DICTA_Shared_Backbone' in str(Path.cwd()):
    STRICT_CKPT_PATH = PROJECT_ROOT / 'models' / 'trained' / 'dicta_shared_backbone' / 'best_dicta_model.pth'
    STRICT_METRICS_OUT = STRICT_CKPT_PATH.parent / 'sota_thermalfusion_net_metrics.json'

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'scripts' / 'data'))
sys.path.insert(0, str(PROJECT_ROOT / 'models' / 'sota_baselines'))

from flame_dataset import FLAMEDataset
from thermalfusion_net_2023 import create_thermalfusion_net_2023


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def fingerprint(obj) -> str:
    payload = json.dumps(obj, sort_keys=True, separators=(',', ':'), ensure_ascii=True)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def to_repo_rel_str(path_obj: Path) -> str:
    return path_obj.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()


def build_or_load_locked_split(sample_keys):
    STRICT_SPLIT_FILE.parent.mkdir(parents=True, exist_ok=True)

    if STRICT_SPLIT_FILE.exists():
        with open(STRICT_SPLIT_FILE, 'r', encoding='utf-8') as f:
            payload = json.load(f)
        source = 'locked-existing'
    else:
        n = len(sample_keys)
        rng = np.random.default_rng(SEED)
        perm = rng.permutation(n).tolist()
        train_end = int(0.8 * n)
        val_end = int(0.9 * n)

        payload = {
            'version': 1,
            'seed': SEED,
            'num_samples': n,
            'ratios': {'train': 0.8, 'val': 0.1, 'test': 0.1},
            'train_idx': perm[:train_end],
            'val_idx': perm[train_end:val_end],
            'test_idx': perm[val_end:],
            'dataset_root': 'data/processed/Output/Segmentation_Augmented',
            'image_size': IMAGE_SIZE,
            'normalization': {
                'mean': NORMALIZATION_MEAN,
                'std': NORMALIZATION_STD,
            },
            'sample_count': n,
            'sample_keys': sample_keys,
        }
        payload['split_signature'] = fingerprint({
            'train_idx': payload['train_idx'],
            'val_idx': payload['val_idx'],
            'test_idx': payload['test_idx'],
        })
        payload['sample_signature'] = fingerprint(sample_keys)

        with open(STRICT_SPLIT_FILE, 'w', encoding='utf-8') as f:
            json.dump(payload, f, indent=2)
        source = 'created-once'

    train_idx = [int(i) for i in payload['train_idx']]
    val_idx = [int(i) for i in payload['val_idx']]
    test_idx = [int(i) for i in payload['test_idx']]

    computed_split_signature = fingerprint({'train_idx': train_idx, 'val_idx': val_idx, 'test_idx': test_idx})
    stored_split_signature = payload.get('split_signature', '<missing>')

    split_keys = payload.get('sample_keys')
    if split_keys is not None and list(split_keys) != list(sample_keys):
        raise RuntimeError('Locked split sample ordering mismatch. Dataset differs from locked reference.')

    computed_sample_signature = fingerprint(sample_keys)
    stored_sample_signature = payload.get('sample_signature', computed_sample_signature)

    return {
        'train_idx': train_idx,
        'val_idx': val_idx,
        'test_idx': test_idx,
        'source': source,
        'stored_split_signature': stored_split_signature,
        'computed_split_signature': computed_split_signature,
        'stored_sample_signature': stored_sample_signature,
        'computed_sample_signature': computed_sample_signature,
    }


def benchmark_latency_ms(model, input_shape=(1, 3, 256, 256), warmup=20, runs=100):
    x = torch.randn(*input_shape, device=device)
    times = []

    model.eval()
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(x)

        for _ in range(runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(x)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append((t1 - t0) * 1000.0)

    arr = np.array(times)
    return {
        'mean_ms': float(arr.mean()),
        'p95_ms': float(np.percentile(arr, 95)),
        'p99_ms': float(np.percentile(arr, 99)),
    }


strict_full_dataset = FLAMEDataset(
    root_dir=DATASET_ROOT,
    split='full',
    img_size=IMAGE_SIZE,
    augment=False,
    seed=SEED,
)

sample_keys = []
for img_path in strict_full_dataset.image_files:
    mask_path = strict_full_dataset._get_mask_file(img_path)
    sample_keys.append(f'{to_repo_rel_str(img_path)}|{to_repo_rel_str(mask_path)}')

split_info = build_or_load_locked_split(sample_keys)
train_idx = split_info['train_idx']
val_idx = split_info['val_idx']
test_idx = split_info['test_idx']

n_total = len(strict_full_dataset)
all_idx = set(range(n_total))
train_set_idx = set(train_idx)
val_set_idx = set(val_idx)
test_set_idx = set(test_idx)
no_leak = train_set_idx.isdisjoint(val_set_idx) and train_set_idx.isdisjoint(test_set_idx) and val_set_idx.isdisjoint(test_set_idx)
covered_all = (train_set_idx | val_set_idx | test_set_idx) == all_idx

if not no_leak:
    raise RuntimeError('Data leakage detected: train/val/test splits overlap.')
if not covered_all:
    raise RuntimeError('Split indices do not cover the full dataset exactly once.')

strict_test_dataset = Subset(strict_full_dataset, test_idx)
strict_test_loader = DataLoader(
    strict_test_dataset,
    batch_size=BATCH_SIZE_STRICT,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)

strict_model = create_thermalfusion_net_2023(in_channels=3, out_channels=1, base_channels=12).to(device)
if not STRICT_CKPT_PATH.exists():
    raise FileNotFoundError(f'Checkpoint not found: {STRICT_CKPT_PATH}')

strict_checkpoint = torch.load(STRICT_CKPT_PATH, map_location=device, weights_only=False)
state_dict = strict_checkpoint['model_state'] if isinstance(strict_checkpoint, dict) and 'model_state' in strict_checkpoint else strict_checkpoint
strict_model.load_state_dict(state_dict)
strict_model.eval()

# Full-test aggregate confusion totals
tp = fp = fn = tn = 0
with torch.no_grad():
    for batch in strict_test_loader:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)

        pred = strict_model(images)
        pred_bin = (pred >= 0.5).to(torch.int64)
        true_bin = (masks >= 0.5).to(torch.int64)

        tp += int(((pred_bin == 1) & (true_bin == 1)).sum().item())
        fp += int(((pred_bin == 1) & (true_bin == 0)).sum().item())
        fn += int(((pred_bin == 0) & (true_bin == 1)).sum().item())
        tn += int(((pred_bin == 0) & (true_bin == 0)).sum().item())

eps = 1e-8
precision = tp / (tp + fp + eps)
recall = tp / (tp + fn + eps)
f1 = (2 * precision * recall) / (precision + recall + eps)
fire_iou = tp / (tp + fp + fn + eps)
iou_bg = tn / (tn + fp + fn + eps)
miou = 0.5 * (iou_bg + fire_iou)
fire_f1 = f1
pixel_accuracy = (tp + tn) / (tp + tn + fp + fn + eps)

parameter_count = int(sum(p.numel() for p in strict_model.parameters()))
model_size_mb = float(parameter_count * 4 / (1024 ** 2))
latency_stats = benchmark_latency_ms(strict_model, input_shape=(1, 3, IMAGE_SIZE, IMAGE_SIZE), warmup=20, runs=100)
fps = 1000.0 / max(latency_stats['mean_ms'], eps)

strict_test_metrics = {
    'miou': float(miou),
    'fire_iou': float(fire_iou),
    'fire_f1': float(fire_f1),
    'accuracy': float(pixel_accuracy),
    'classification_accuracy': float(pixel_accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1': float(f1),
    'false_positives_pixels': int(fp),
    'false_negatives_pixels': int(fn),
    'true_positives_pixels': int(tp),
    'true_negatives_pixels': int(tn),
    'parameter_count': parameter_count,
    'model_size_mb': float(model_size_mb),
    'latency_mean_ms': float(latency_stats['mean_ms']),
    'latency_p95_ms': float(latency_stats['p95_ms']),
    'fps': float(fps),
}

print('=' * 110)
print('STRICT TEST-SET EVALUATION (LOCKED SPLIT, MACHINE-AGNOSTIC)')
print('=' * 110)
print('Device:', device)
print('Project root:', PROJECT_ROOT)
print('Dataset root:', DATASET_ROOT)
print('Locked split file:', STRICT_SPLIT_FILE)
print('Checkpoint file:', STRICT_CKPT_PATH)
print('Locked split source:', split_info['source'])
print('Stored split signature:', split_info['stored_split_signature'])
print('Computed split signature:', split_info['computed_split_signature'])
print('Stored sample signature:', split_info['stored_sample_signature'])
print('Computed sample signature:', split_info['computed_sample_signature'])
print(f'Sample counts -> train: {len(train_idx)} | val: {len(val_idx)} | test: {len(test_idx)} | total: {n_total}')
print('Disjoint train/val/test:', no_leak)
print('Full coverage exactly once:', covered_all)
print('-' * 110)
print('Final locked-test metrics (full-test aggregate):')
print(json.dumps(strict_test_metrics, indent=2))

strict_metrics_payload = {}
if STRICT_METRICS_OUT.exists():
    with open(STRICT_METRICS_OUT, 'r', encoding='utf-8') as f:
        try:
            strict_metrics_payload = json.load(f)
        except json.JSONDecodeError:
            strict_metrics_payload = {}

strict_metrics_payload['strict_test_evaluation'] = {
    'split': {
        'source': split_info['source'],
        'locked_split_file': str(STRICT_SPLIT_FILE),
        'train_samples': len(train_idx),
        'val_samples': len(val_idx),
        'test_samples': len(test_idx),
        'total_samples': n_total,
        'disjoint': no_leak,
        'full_coverage': covered_all,
        'stored_split_signature': split_info['stored_split_signature'],
        'computed_split_signature': split_info['computed_split_signature'],
        'stored_sample_signature': split_info['stored_sample_signature'],
        'computed_sample_signature': split_info['computed_sample_signature'],
    },
    'metrics': strict_test_metrics,
    'checkpoint_path': str(STRICT_CKPT_PATH),
    'dataset_root': str(DATASET_ROOT),
    'image_size': IMAGE_SIZE,
}

STRICT_METRICS_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(STRICT_METRICS_OUT, 'w', encoding='utf-8') as f:
    json.dump(strict_metrics_payload, f, indent=2)

print(f'Strict test metrics appended to: {STRICT_METRICS_OUT}')

FLAME Dataset (full): 808 samples, size=256x256, augment=False
STRICT TEST-SET EVALUATION (LOCKED SPLIT, MACHINE-AGNOSTIC)
Device: cpu
Project root: c:\SPJAIN\BushFire-Detection
Dataset root: c:\SPJAIN\BushFire-Detection\data\processed\Output\Segmentation_Augmented
Locked split file: c:\SPJAIN\BushFire-Detection\data\processed\Output\DICTA\flame_strict_split_seed42.json
Checkpoint file: c:\SPJAIN\BushFire-Detection\models\trained\sota_baselines\thermalfusion_net_2023\best_model.pth
Locked split source: locked-existing
Stored split signature: 9158fd7a915daf798392349c0b6c781b6e89fe6d468db4785026eac89bed695b
Computed split signature: d85b0056df5e3aedf0a24aa02fc24cf965eb28cc4625ae4c06bda76d4ccfe303
Stored sample signature: a08d2599bd3a679fa5763e05910d48bc7a6ab949491329f2737856e24d3c1ab9
Computed sample signature: a08d2599bd3a679fa5763e05910d48bc7a6ab949491329f2737856e24d3c1ab9
Sample counts -> train: 646 | val: 80 | test: 82 | total: 808
Disjoint train/val/test: True
Full coverage exactly 